In [1]:
import os
import glob
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import torch.nn as nn

# Base path — local machine
BASE_DIR        = r'C:\Users\elent\Downloads\Eye Disease Detection'
DATASET_DIR     = os.path.join(BASE_DIR, 'dataset', 'OCT2017')
TRAIN_DIR       = os.path.join(DATASET_DIR, 'train')
VAL_DIR         = os.path.join(DATASET_DIR, 'val')
TEST_DIR        = os.path.join(DATASET_DIR, 'test')
FEATURES_DIR    = os.path.join(BASE_DIR, 'features')
CHECKPOINTS_DIR = os.path.join(BASE_DIR, 'checkpoints')
RESULTS_DIR     = os.path.join(BASE_DIR, 'results')
CLASSES         = ['CNV', 'DME', 'DRUSEN', 'NORMAL']

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")

# Verify paths
for name, path in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    exists = os.path.exists(path)
    print(f"{name}: {'✅ Found' if exists else '❌ NOT FOUND'}")

✅ Using device: cuda
✅ GPU: NVIDIA GeForce RTX 5060 Laptop GPU
Train: ✅ Found
Val: ✅ Found
Test: ✅ Found


In [2]:
# Load pretrained ResNet50
resnet50 = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Remove final classification layer
# We want 2048-d feature vectors not 1000 class predictions
resnet50.fc = nn.Identity()

# Freeze all layers
for param in resnet50.parameters():
    param.requires_grad = False

# Move to GPU and set to eval mode
resnet50 = resnet50.to(device)
resnet50.eval()

print("✅ ResNet50 loaded and ready")
print("✅ All layers frozen")
print("✅ Final layer replaced with Identity (outputs 2048-d vector)")
print(f"✅ Model moved to {device}")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\elent/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:02<00:00, 39.8MB/s]


✅ ResNet50 loaded and ready
✅ All layers frozen
✅ Final layer replaced with Identity (outputs 2048-d vector)
✅ Model moved to cuda


In [3]:
# Transform pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Custom Dataset class
class OCTDataset(Dataset):
    def __init__(self, split_dir, classes, transform=None):
        self.transform = transform
        self.image_paths = []
        self.labels = []

        for label_idx, cls in enumerate(classes):
            cls_path = os.path.join(split_dir, cls)
            images = []
            for ext in ['*.jpeg', '*.jpg', '*.JPEG', '*.JPG', '*.png', '*.PNG']:
                images.extend(glob.glob(os.path.join(cls_path, ext)))
            images = list(set(images))

            for img_path in images:
                self.image_paths.append(img_path)
                self.labels.append(label_idx)

        print(f"  Loaded {len(self.image_paths)} images from {split_dir.split(os.sep)[-1]}")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        try:
            img = Image.open(img_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
        except Exception as e:
            img = torch.zeros(3, 224, 224)
        return img, label, img_path

# Create datasets
print("📂 Loading datasets...")
train_dataset = OCTDataset(TRAIN_DIR, CLASSES, transform)
val_dataset   = OCTDataset(VAL_DIR,   CLASSES, transform)
test_dataset  = OCTDataset(TEST_DIR,  CLASSES, transform)

print(f"\n✅ Total images ready:")
print(f"   Train : {len(train_dataset)}")
print(f"   Val   : {len(val_dataset)}")
print(f"   Test  : {len(test_dataset)}")

📂 Loading datasets...
  Loaded 83484 images from train
  Loaded 32 images from val
  Loaded 968 images from test

✅ Total images ready:
   Train : 83484
   Val   : 32
   Test  : 968


In [4]:
def extract_and_save(dataset, model, batch_size=64, split_name='', save_dir=''):
    features_path = os.path.join(save_dir, f'{split_name}_features.npy')
    labels_path   = os.path.join(save_dir, f'{split_name}_labels.npy')
    paths_path    = os.path.join(save_dir, f'{split_name}_paths.npy')

    # If already extracted just load them
    if os.path.exists(features_path) and os.path.exists(labels_path):
        print(f"✅ {split_name} features already exist — loading...")
        features = np.load(features_path)
        labels   = np.load(labels_path)
        paths    = np.load(paths_path)
        print(f"   Shape: {features.shape}")
        return features, labels, paths

    # Otherwise extract
    dataloader = DataLoader(dataset, batch_size=batch_size,
                           shuffle=False, num_workers=0, pin_memory=True)

    all_features = []
    all_labels   = []
    all_paths    = []

    print(f"\n🔄 Extracting features for {split_name}...")
    print(f"   Total images : {len(dataset)}")
    print(f"   Batch size   : {batch_size}")
    print(f"   Total batches: {len(dataloader)}")

    with torch.no_grad():
        for batch_imgs, batch_labels, batch_paths in tqdm(dataloader, desc=split_name):
            batch_imgs = batch_imgs.to(device)
            features   = model(batch_imgs)

            all_features.append(features.cpu().numpy())
            all_labels.extend(batch_labels.numpy())
            all_paths.extend(batch_paths)

    all_features = np.concatenate(all_features, axis=0)
    all_labels   = np.array(all_labels)

    # Save to features folder
    np.save(features_path, all_features)
    np.save(labels_path,   all_labels)
    np.save(paths_path,    np.array(all_paths))

    print(f"   ✅ Saved to {save_dir}")
    print(f"   Shape: {all_features.shape}")
    return all_features, all_labels, all_paths

print("✅ Function defined")

✅ Function defined


In [5]:
train_features, train_labels, train_paths = extract_and_save(
    train_dataset, resnet50, batch_size=64,
    split_name='train', save_dir=FEATURES_DIR)

val_features, val_labels, val_paths = extract_and_save(
    val_dataset, resnet50, batch_size=64,
    split_name='val', save_dir=FEATURES_DIR)

test_features, test_labels, test_paths = extract_and_save(
    test_dataset, resnet50, batch_size=64,
    split_name='test', save_dir=FEATURES_DIR)

print("\n🎉 All done!")
print(f"   Train : {train_features.shape}")
print(f"   Val   : {val_features.shape}")
print(f"   Test  : {test_features.shape}")


🔄 Extracting features for train...
   Total images : 83484
   Batch size   : 64
   Total batches: 1305


train: 100%|███████████████████████████████████████████████████████████████████████| 1305/1305 [08:08<00:00,  2.67it/s]


   ✅ Saved to C:\Users\elent\Downloads\Eye Disease Detection\features
   Shape: (83484, 2048)

🔄 Extracting features for val...
   Total images : 32
   Batch size   : 64
   Total batches: 1


val: 100%|███████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.95it/s]


   ✅ Saved to C:\Users\elent\Downloads\Eye Disease Detection\features
   Shape: (32, 2048)

🔄 Extracting features for test...
   Total images : 968
   Batch size   : 64
   Total batches: 16


test: 100%|████████████████████████████████████████████████████████████████████████████| 16/16 [00:05<00:00,  3.14it/s]

   ✅ Saved to C:\Users\elent\Downloads\Eye Disease Detection\features
   Shape: (968, 2048)

🎉 All done!
   Train : (83484, 2048)
   Val   : (32, 2048)
   Test  : (968, 2048)


In [6]:
import numpy as np
import os

print("🔍 Verifying saved feature files...\n")

splits = ['train', 'val', 'test']

for split in splits:
    features_path = os.path.join(FEATURES_DIR, f'{split}_features.npy')
    labels_path   = os.path.join(FEATURES_DIR, f'{split}_labels.npy')
    paths_path    = os.path.join(FEATURES_DIR, f'{split}_paths.npy')

    features = np.load(features_path)
    labels   = np.load(labels_path)
    paths    = np.load(paths_path)

    print(f"📂 {split.upper()}")
    print(f"   Features shape : {features.shape}")
    print(f"   Labels shape   : {labels.shape}")
    print(f"   Paths shape    : {paths.shape}")
    print(f"   Feature range  : min={features.min():.4f}, max={features.max():.4f}")
    print(f"   Unique labels  : {sorted(set(labels.tolist()))} (0=CNV, 1=DME, 2=DRUSEN, 3=NORMAL)")
    
    # Count per class
    for idx, cls in enumerate(CLASSES):
        count = (labels == idx).sum()
        print(f"   {cls:<10}: {count} samples")
    print()

print("✅ All feature files verified and ready for Notebook 3!")

🔍 Verifying saved feature files...

📂 TRAIN
   Features shape : (83484, 2048)
   Labels shape   : (83484,)
   Paths shape    : (83484,)
   Feature range  : min=0.0000, max=9.6612
   Unique labels  : [0, 1, 2, 3] (0=CNV, 1=DME, 2=DRUSEN, 3=NORMAL)
   CNV       : 37205 samples
   DME       : 11348 samples
   DRUSEN    : 8616 samples
   NORMAL    : 26315 samples

📂 VAL
   Features shape : (32, 2048)
   Labels shape   : (32,)
   Paths shape    : (32,)
   Feature range  : min=0.0000, max=7.8596
   Unique labels  : [0, 1, 2, 3] (0=CNV, 1=DME, 2=DRUSEN, 3=NORMAL)
   CNV       : 8 samples
   DME       : 8 samples
   DRUSEN    : 8 samples
   NORMAL    : 8 samples

📂 TEST
   Features shape : (968, 2048)
   Labels shape   : (968,)
   Paths shape    : (968,)
   Feature range  : min=0.0000, max=8.6365
   Unique labels  : [0, 1, 2, 3] (0=CNV, 1=DME, 2=DRUSEN, 3=NORMAL)
   CNV       : 242 samples
   DME       : 242 samples
   DRUSEN    : 242 samples
   NORMAL    : 242 samples

✅ All feature files ver